# Project 2

## Configuration

In [1]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, TimestampType
)
# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
CHECKPOINT_BASE = "/home/jovyan/project/checkpoints"

BRONZE_CHECKPOINT  = f"{CHECKPOINT_BASE}/bronze"
SILVER_CHECKPOINT  = f"{CHECKPOINT_BASE}/silver"
GOLD_CHECKPOINT    = f"{CHECKPOINT_BASE}/gold"
METRICS_CHECKPOINT = f"{CHECKPOINT_BASE}/metrics"

TRIGGER_INTERVAL = "5 seconds"

In [4]:
# Cleanup
'''
import shutil
shutil.rmtree(BRONZE_CHECKPOINT, ignore_errors=True)
shutil.rmtree(SILVER_CHECKPOINT, ignore_errors=True)
shutil.rmtree(GOLD_CHECKPOINT, ignore_errors=True)
shutil.rmtree(METRICS_CHECKPOINT, ignore_errors=True)
spark.sql("TRUNCATE TABLE lakehouse.taxi.bronze_trips")
spark.sql("TRUNCATE TABLE lakehouse.taxi.silver_trips")
spark.sql("TRUNCATE TABLE lakehouse.taxi.gold_route_stats")
spark.sql("TRUNCATE TABLE lakehouse.taxi.metrics")
'''

'\nimport shutil\nshutil.rmtree(BRONZE_CHECKPOINT, ignore_errors=True)\nshutil.rmtree(SILVER_CHECKPOINT, ignore_errors=True)\nshutil.rmtree(GOLD_CHECKPOINT, ignore_errors=True)\nshutil.rmtree(METRICS_CHECKPOINT, ignore_errors=True)\nspark.sql("TRUNCATE TABLE lakehouse.taxi.bronze_trips")\nspark.sql("TRUNCATE TABLE lakehouse.taxi.silver_trips")\nspark.sql("TRUNCATE TABLE lakehouse.taxi.gold_route_stats")\nspark.sql("TRUNCATE TABLE lakehouse.taxi.metrics")\n'

## Bronze layer

In [5]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_trips (
    key STRING,
    value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP
)
USING iceberg
""")

bronze_input = (
    raw_stream
    .select(
        F.col("key").cast("string"),
        F.col("value").cast("string"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp")
    )
)

bronze_query = (
    bronze_input.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable("lakehouse.taxi.bronze_trips")
)

# Silver layer

In [6]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver_trips (
    VendorID             INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count      INT,
    trip_distance        DOUBLE,
    PULocationID         INT,
    DOLocationID         INT,
    fare_amount          DOUBLE,
    total_amount         DOUBLE,
    pickup_borough       STRING,
    pickup_zone          STRING,
    pickup_service_zone  STRING,
    dropoff_borough      STRING,
    dropoff_zone         STRING,
    dropoff_service_zone STRING
)
USING ICEBERG
PARTITIONED BY (days(tpep_pickup_datetime))
""")

schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("total_amount",          DoubleType()),
])

pu = zones.alias("pu")
do = zones.alias("do")

parsed_stream = (
    raw_stream
    .select(
        F.from_json(F.col("value").cast("string"), schema).alias("json")
    )
    .select("json.*")
)

cleaned_stream = (
    parsed_stream
    .select(
        F.col("VendorID").cast("int"),
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime"),
        F.when(
            (F.col("passenger_count").isNull()) | (F.col("passenger_count") <= 0), 1
        ).otherwise(F.col("passenger_count")).cast("int").alias("passenger_count"),
        F.col("trip_distance").cast("double"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        F.col("fare_amount").cast("double"),
        F.col("total_amount").cast("double"),
    )
    .dropna(subset=[
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "PULocationID", "DOLocationID"
    ])
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter(F.col("trip_distance") > 0)
    .filter((F.col("passenger_count") > 0) & (F.col("passenger_count") < 10))
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("total_amount") >= 0)
    .withWatermark("tpep_pickup_datetime", "1 day")
    .dropDuplicates([
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount"
    ])
)

silver_input = (

    cleaned_stream
    .join(F.broadcast(pu), cleaned_stream.PULocationID == F.col("pu.LocationID"), "left")
    .join(F.broadcast(do), cleaned_stream.DOLocationID == F.col("do.LocationID"), "left")
    .select(
        cleaned_stream["*"],
        F.col("pu.Borough").alias("pickup_borough"),
        F.col("pu.Zone").alias("pickup_zone"),
        F.col("pu.service_zone").alias("pickup_service_zone"),
        F.col("do.Borough").alias("dropoff_borough"),
        F.col("do.Zone").alias("dropoff_zone"),
        F.col("do.service_zone").alias("dropoff_service_zone"),
    )
    .filter(
        F.col("pickup_zone").isNotNull() &
        F.col("dropoff_zone").isNotNull()
    )
)

silver_query = (
    silver_input.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable("lakehouse.taxi.silver_trips")
)

# Gold layer

In [7]:

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold_route_stats (
    pickup_borough    STRING,
    dropoff_borough   STRING,
    trip_count        BIGINT,
    total_revenue     DOUBLE,
    avg_distance      DOUBLE,
    revenue_per_mile  DOUBLE
)
USING iceberg
PARTITIONED BY (pickup_borough)
""")


DataFrame[]

In [8]:
def update_gold_route_stats():
    full_silver_df = spark.table("lakehouse.taxi.silver_trips")

    gold_df = (
        full_silver_df
        .groupBy("pickup_borough", "dropoff_borough")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(
                F.when(
                    F.sum("trip_distance") > 0,
                    F.sum("total_amount") / F.sum("trip_distance")
                ).otherwise(0.0),
                2
            ).alias("revenue_per_mile")
        )
    )

    gold_df.writeTo("lakehouse.taxi.gold_route_stats").overwritePartitions()

# Custom scenario

In [9]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.metrics (
    batch_id        BIGINT,
    topic           STRING,
    partition       INT,
    starting_offset BIGINT,
    ending_offset   BIGINT,
    rows_processed  BIGINT,
    batch_timestamp TIMESTAMP
)
USING iceberg
""")

DataFrame[]

In [10]:
def write_batch_metrics(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    metrics_df = (
        batch_df
        .groupBy("topic", "partition")
        .agg(
            F.min("offset").alias("starting_offset"),
            F.max("offset").alias("ending_offset"),
            F.count("*").alias("rows_processed")
        )
        .withColumn("batch_id", F.lit(batch_id).cast("bigint"))
        .withColumn("batch_timestamp", F.current_timestamp())
        .select(
            "batch_id",
            "topic",
            "partition",
            "starting_offset",
            "ending_offset",
            "rows_processed",
            "batch_timestamp"
        )
    )

    metrics_df.writeTo("lakehouse.taxi.metrics").append()

In [11]:
metrics_query = (
    bronze_input.writeStream
    .foreachBatch(write_batch_metrics)
    .option("checkpointLocation", METRICS_CHECKPOINT)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)

# Pre-verification/ pipeline status

#### Cells from here onwards should be run once the producer has been started and data-flow is active

In [12]:
update_gold_route_stats() # rerun this cell to get updated stats

In [13]:
print(len(spark.streams.active))
for q in spark.streams.active:
    print(q.isActive, q.status, q.exception())

3
True {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False} None
True {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False} None
True {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False} None


In [14]:
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.bronze_trips").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.silver_trips").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.metrics").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.gold_route_stats").show()
spark.table("lakehouse.taxi.gold_route_stats").sort(F.desc("trip_count")).show()

+--------+
|count(1)|
+--------+
|   13199|
+--------+

+--------+
|count(1)|
+--------+
|   12313|
+--------+

+--------+
|count(1)|
+--------+
|      68|
+--------+

+--------+
|count(1)|
+--------+
|      28|
+--------+

+--------------+---------------+----------+-------------+------------+----------------+
|pickup_borough|dropoff_borough|trip_count|total_revenue|avg_distance|revenue_per_mile|
+--------------+---------------+----------+-------------+------------+----------------+
|     Manhattan|      Manhattan|     10736|    246838.32|        2.21|           10.42|
|     Manhattan|       Brooklyn|       504|     22728.29|        6.49|            6.94|
|     Manhattan|         Queens|       266|     11988.19|        7.23|            6.24|
|        Queens|       Brooklyn|       146|      9564.15|       12.84|             5.1|
|        Queens|      Manhattan|       142|     11268.39|       13.83|            5.74|
|        Queens|         Queens|       132|      4464.44|        5.95|  

# Verification

In [15]:
# --- Bronze layer stuff ---
time.sleep(12)

count_before = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Bronze query stopped.")

# Restart using the SAME checkpoint directory.
bronze_query = (
    bronze_input.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)

count_after = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()

if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — those are new messages produced between the two runs.")

bronze_audit = spark.table("lakehouse.taxi.bronze_trips")

bronze_audit.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        F.concat_ws(
            ":",
            F.col("topic"),
            F.col("partition").cast("string"),
            F.col("offset").cast("string")
        )
    ).alias("unique_kafka_messages")
).show()

Row count before restart: 13199
Bronze query stopped.
Row count before restart : 13199
Row count after restart  : 15179

Counts differ by 1980 — those are new messages produced between the two runs.
+----------+---------------------+
|total_rows|unique_kafka_messages|
+----------+---------------------+
|     15575|                15575|
+----------+---------------------+



In [16]:
spark.sql("""
SELECT count(*) FROM lakehouse.taxi.bronze_trips;
""").show()

spark.sql("""
SELECT * FROM lakehouse.taxi.bronze_trips LIMIT 3;
""").show()

+--------+
|count(1)|
+--------+
|   15575|
+--------+

+---+--------------------+----------+---------+------+--------------------+
|key|               value|     topic|partition|offset|           timestamp|
+---+--------------------+----------+---------+------+--------------------+
|  1|{"VendorID": 1, "...|taxi-trips|        0|  1007|2026-04-05 20:25:...|
|  1|{"VendorID": 1, "...|taxi-trips|        0|  1008|2026-04-05 20:25:...|
|  1|{"VendorID": 1, "...|taxi-trips|        0|  1009|2026-04-05 20:25:...|
+---+--------------------+----------+---------+------+--------------------+



In [17]:
# --Silver Layer Stuff---
bronze_count = spark.table("lakehouse.taxi.bronze_trips").count()
silver_count = spark.table("lakehouse.taxi.silver_trips").count()
print(f"Bronze rows : {bronze_count}")
print(f"Silver rows : {silver_count}")
print(f"Drop rate   : {1 - silver_count/bronze_count:.1%}  (filtered/invalid records)\n")

Bronze rows : 15575
Silver rows : 12313
Drop rate   : 20.9%  (filtered/invalid records)



In [18]:
silver_check = spark.read.table("lakehouse.taxi.silver_trips")
silver_check.show(5, truncate=False)
print(f"Silver count: {silver_check.count()}\n")

print("Null counts per column")
silver_check.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_check.columns
]).show()

print("Trip distribution by borough")
silver_check.groupBy("pickup_borough").count().show()

print("Trip duration statistics")
silver_check.select(
    (F.col("tpep_dropoff_datetime").cast("long") - 
     F.col("tpep_pickup_datetime").cast("long")).alias("duration_sec")
).describe().show()

print("Total vs unique")
spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT *) AS unique_rows
FROM lakehouse.taxi.silver_trips;""").show()

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+---------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|pickup_borough|pickup_zone                  |pickup_service_zone|dropoff_borough|dropoff_zone   |dropoff_service_zone|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+---------------+--------------------+
|2       |2024-12-31 23:37:42 |2024-12-31 23:43:10  |1              |0.92         |229         |141         |7.2        |14.03       |Manhattan     |Sutton Place/Turtle Bay North|Yellow Zone        |Manhattan      |Lenox Hill West|Yellow Zon

#### Verify, that restart does not produce duplicates

In [19]:
# Gold stuff
time.sleep(12)

gold_count = spark.table("lakehouse.taxi.gold_route_stats").count()
print(f"Gold row count: {gold_count}")

spark.table("lakehouse.taxi.gold_route_stats").sort(F.desc("trip_count")).show()

Gold row count: 28
+--------------+---------------+----------+-------------+------------+----------------+
|pickup_borough|dropoff_borough|trip_count|total_revenue|avg_distance|revenue_per_mile|
+--------------+---------------+----------+-------------+------------+----------------+
|     Manhattan|      Manhattan|     10736|    246838.32|        2.21|           10.42|
|     Manhattan|       Brooklyn|       504|     22728.29|        6.49|            6.94|
|     Manhattan|         Queens|       266|     11988.19|        7.23|            6.24|
|        Queens|       Brooklyn|       146|      9564.15|       12.84|             5.1|
|        Queens|      Manhattan|       142|     11268.39|       13.83|            5.74|
|        Queens|         Queens|       132|      4464.44|        5.95|            5.69|
|     Manhattan|          Bronx|        67|      3257.86|        9.09|            5.35|
|      Brooklyn|      Manhattan|        67|      2803.58|        5.85|            7.16|
|      Brookl

In [20]:
gold_audit_df = spark.sql("""
    SELECT 
        snapshot_id,
        committed_at,
        operation,
        summary['replace-partitions'] AS is_overwrite,
        summary['added-records']      AS rows_added,
        summary['deleted-records']    AS rows_deleted,
        summary['total-records']      AS total_table_rows,
        summary['added-data-files']   AS files_added,
        summary['deleted-data-files'] AS files_deleted
    FROM lakehouse.taxi.gold_route_stats.snapshots
    ORDER BY committed_at DESC
""")

gold_audit_df.show(truncate=False)

+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+
|snapshot_id        |committed_at           |operation|is_overwrite|rows_added|rows_deleted|total_table_rows|files_added|files_deleted|
+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+
|5132488330274879044|2026-04-05 20:26:42.565|overwrite|true        |28        |NULL        |28              |6          |NULL         |
+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+



# Custom scenario metrics

In [21]:
time.sleep(12)

spark.sql("""
SELECT *
FROM lakehouse.taxi.metrics
ORDER BY batch_timestamp DESC, partition
""").show(50, truncate=False)

+--------+----------+---------+---------------+-------------+--------------+--------------------------+
|batch_id|topic     |partition|starting_offset|ending_offset|rows_processed|batch_timestamp           |
+--------+----------+---------+---------------+-------------+--------------+--------------------------+
|45      |taxi-trips|0        |3378           |3482         |105           |2026-04-05 20:27:40.126751|
|45      |taxi-trips|2        |13783          |14075        |293           |2026-04-05 20:27:40.126751|
|44      |taxi-trips|0        |3295           |3377         |83            |2026-04-05 20:27:35.145866|
|44      |taxi-trips|2        |13470          |13782        |313           |2026-04-05 20:27:35.145866|
|43      |taxi-trips|0        |3201           |3294         |94            |2026-04-05 20:27:30.102405|
|43      |taxi-trips|2        |13176          |13469        |294           |2026-04-05 20:27:30.102405|
|42      |taxi-trips|0        |3138           |3200         |63 

In [22]:
spark.sql("""
SELECT
    batch_id,
    topic,
    partition,
    starting_offset,
    ending_offset,
    rows_processed,
    (ending_offset - starting_offset + 1) AS offset_span,
    batch_timestamp
FROM lakehouse.taxi.metrics
ORDER BY batch_id, partition
""").show(100, truncate=False)

+--------+----------+---------+---------------+-------------+--------------+-----------+--------------------------+
|batch_id|topic     |partition|starting_offset|ending_offset|rows_processed|offset_span|batch_timestamp           |
+--------+----------+---------+---------------+-------------+--------------+-----------+--------------------------+
|1       |taxi-trips|0        |0              |71           |72            |72         |2026-04-05 20:24:01.075171|
|1       |taxi-trips|2        |0              |131          |132           |132        |2026-04-05 20:24:01.075171|
|2       |taxi-trips|0        |72             |159          |88            |88         |2026-04-05 20:24:05.223573|
|2       |taxi-trips|2        |132            |412          |281           |281        |2026-04-05 20:24:05.223573|
|3       |taxi-trips|0        |160            |246          |87            |87         |2026-04-05 20:24:10.144187|
|3       |taxi-trips|2        |413            |723          |311        